In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import base64, io, xarray as xr

ds = xr.open_dataset("icon_ch1_TOT_PREC_all_lead_times.nc")
hourly_rain = ds["hourly_rain"]
max_rain = hourly_rain.max(dim="lead_time").values

CH_LAT, CH_LON = 46.8, 8.2
lons = np.linspace(-0.817, 18.183, 429)
lats = np.linspace(41.183, 51.183, 295)
LON, LAT = np.meshgrid(lons, lats)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = np.radians(lat2-lat1), np.radians(lon2-lon1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

within_mask = haversine(LAT, LON, CH_LAT, CH_LON) <= 350

max_val = np.nanmax(max_rain)
norm = mcolors.Normalize(vmin=0, vmax=max_val)
cmap = plt.cm.YlOrRd
max_rain_masked = np.where(within_mask, max_rain, np.nan)
rgba = cmap(norm(np.nan_to_num(max_rain_masked)))
rgba[..., 3] = np.where(within_mask & (max_rain_masked >= 0.01), 1.0, 0)
rgba_flipped = np.flipud(rgba)

buf = io.BytesIO()
plt.imsave(buf, rgba_flipped, format="png")
buf.seek(0)
data_uri = "data:image/png;base64," + base64.b64encode(buf.read()).decode("utf-8")

html = f"""<!DOCTYPE html>
<html><head>
<meta charset="UTF-8">
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  * {{ margin:0; padding:0; box-sizing:border-box; }}
  #map {{ width:100vw; height:100vh; }}
  #controls {{
    position:absolute; top:12px; left:50%; transform:translateX(-50%);
    z-index:1000; background:rgba(20,20,20,0.85); backdrop-filter:blur(8px);
    border-radius:10px; padding:10px 18px; display:flex; align-items:center; gap:16px;
    border:1px solid rgba(255,255,255,0.1); font-family:system-ui,sans-serif;
  }}
  #controls label {{ font-size:13px; color:#aaa; }}
  #opacity-slider {{ width:120px; accent-color:#4f98a3; }}
  #opacity-val {{ font-size:13px; color:#4f98a3; min-width:30px; }}
</style>
</head><body>
<div id="map"></div>
<div id="controls">
  <label>Opacity</label>
  <input type="range" id="opacity-slider" min="0" max="100" value="65">
  <span id="opacity-val">65%</span>
</div>
<script>
  // 1. Map — no default tiles
  const map = L.map('map').setView([46.5, 8.5], 7);

  // 2. Base (no labels)
  L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_nolabels/{{z}}/{{x}}/{{y}}{{r}}.png', {{
    attribution: '© OpenStreetMap contributors © CARTO',
    subdomains: 'abcd', maxZoom: 19
  }}).addTo(map);

  // 3. Rain overlay
  const rainOverlay = L.imageOverlay('{data_uri}',
    [[41.183, -0.817], [51.183, 18.183]],
    {{opacity: 0.65, interactive: false}}
  ).addTo(map);

  // 4. Labels pane — forced above everything
  const labelsPane = map.createPane('labelsPane');
  labelsPane.style.zIndex = 650;
  labelsPane.style.pointerEvents = 'none';
  L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_only_labels/{{z}}/{{x}}/{{y}}{{r}}.png', {{
    subdomains: 'abcd', maxZoom: 19, pane: 'labelsPane'
  }}).addTo(map);

  // Opacity slider
  document.getElementById('opacity-slider').addEventListener('input', function() {{
    rainOverlay.setOpacity(this.value / 100);
    document.getElementById('opacity-val').textContent = this.value + '%';
  }});
</script>
</body></html>"""

with open("/home/ignaz/meteodata-lab-env/rain_leaflet.html", "w") as f:
    f.write(html)
print("Saved! Open: file:///home/ignaz/meteodata-lab-env/rain_leaflet.html")

Saved! Open: file:///home/ignaz/meteodata-lab-env/rain_leaflet.html
